# 👗 StyleVision AI — CatVTON Neural Virtual Try-On Server
### MCA Capstone Project: High-Fidelity Outfit Synthesis on Free NVIDIA T4 GPU

This Google Colab notebook provides **real AI Virtual Try-On inference** powered by **CatVTON** (*Concatenation Is All You Need for Virtual Try-On*).

### ✨ What this does:
1. **Preserves 100% of the User**: Face, hair, body pose, posture, neck, hands, and background remain completely untouched.
2. **Replaces Only the Outfit**: Automatically detects and replaces existing clothing with the chosen saree, gown, dress, or top.
3. **Exposes a Live HTTPS API**: Connects directly to your local StyleVision backend via a free Cloudflare Tunnel.

--- 
### 🚀 Step 1: Ensure GPU is Enabled
Go to **Runtime > Change runtime type > Hardware accelerator > T4 GPU** before running!

In [ ]:
# @title 0. Fix Binary Compatibility (Run ONCE before everything else)
# Run this cell first on a fresh Kaggle session to ensure NumPy/OpenCV are compatible
import subprocess, sys

print('Step 1: Upgrading NumPy + SciPy to 2.x (to match PyTorch compiled headers)...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy', 'scipy'], check=True)

print('Step 2: Reinstalling OpenCV compiled for NumPy 2.x...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--upgrade', 'opencv-python-headless'], check=True)

print('✅ All fixed! Restarting kernel now...')
import os; os._exit(0)  # Auto-restarts the kernel


In [ ]:
# @title 1. Verify NVIDIA GPU Acceleration
!nvidia-smi

import torch
print('='*50)
if torch.cuda.is_available():
    print(f'✅ GPU Detected: {torch.cuda.get_device_name(0)}')
    print(f'⚡ Dedicated VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB')
else:
    print('❌ No GPU detected! Please go to Runtime -> Change runtime type -> Select T4 GPU.')
print('='*50)

### 📦 Step 2: Clone CatVTON & Install Dependencies
Clones the official CatVTON repository, installs diffusion & computer vision requirements, and sets up `cloudflared` for free HTTPS tunneling.

In [ ]:
# @title 2. Clone CatVTON & Install Dependencies (~2 mins)
import os

work_dir = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
catvton_dir = os.path.join(work_dir, 'CatVTON')
if not os.path.exists(catvton_dir):
    print('📥 Cloning CatVTON repository...')
    !git clone https://github.com/Zheng-Chong/CatVTON.git {catvton_dir}

import sys
if catvton_dir not in sys.path:
    sys.path.insert(0, catvton_dir)
os.chdir(catvton_dir)

print('📦 Step 1: Installing core ML packages...')
!pip install -q 'huggingface_hub<0.26.0' 'diffusers==0.27.2' accelerate==0.28.0 transformers==4.39.1 torchvision
!pip install -q fastapi==0.110.0 uvicorn==0.28.0 python-multipart nest_asyncio
!pip install -q gradio==4.21.0

print('🔧 Step 2: Upgrading NumPy + SciPy to 2.x (required by PyTorch on Python 3.13)...')
!pip install -q --upgrade numpy scipy

print('🔧 Step 3: Removing ALL old OpenCV variants (compiled for NumPy 1.x)...')
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless 2>/dev/null || true

print('🔧 Step 4: Installing OpenCV compiled for NumPy 2.x...')
!pip install -q --no-cache-dir 'opencv-python-headless>=4.10.0'

print('🌐 Installing Cloudflare Tunnel (cloudflared)...')
!wget -nc -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('✅ All dependencies installed! Restart the kernel now, then run Cell 3.')


### 🧠 Step 3: Load CatVTON Pipeline & AutoMasker on GPU
Loads the inpainting diffusion model and the SCHP + DensePose segmentation network into CUDA memory.

In [ ]:
# @title 3. Download Model Weights & Load into GPU VRAM (~2 mins)
import sys, types, os

# Restore CatVTON working directory and sys.path after kernel restart
work_dir = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
catvton_dir = os.path.join(work_dir, 'CatVTON')
if catvton_dir not in sys.path:
    sys.path.insert(0, catvton_dir)
os.chdir(catvton_dir)

import numpy as np

# Compatibility shim: restore numpy.core for cv2 compiled against NumPy 1.x
if not hasattr(np, 'core') or not hasattr(getattr(np, 'core', None), 'multiarray'):
    _core_mod = types.ModuleType('numpy.core')
    _core_mod.multiarray = np._core.multiarray
    _core_mod.umath = np._core.umath
    _core_mod.numeric = np._core.numeric
    np.core = _core_mod
    sys.modules['numpy.core'] = _core_mod
    sys.modules['numpy.core.multiarray'] = np._core.multiarray
    sys.modules['numpy.core.umath'] = np._core.umath
    sys.modules['numpy.core.numeric'] = np._core.numeric
    print('✅ NumPy compatibility shim applied for cv2.')

import torch
from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

print('⏳ Downloading CatVTON checkpoints from Hugging Face...')
repo_path = snapshot_download(repo_id='zhengchong/CatVTON')

print('⚡ Initializing CatVTON Diffusion Pipeline on GPU (fp16)...')
pipeline = CatVTONPipeline(
    base_ckpt='runwayml/stable-diffusion-inpainting',
    attn_ckpt=repo_path,
    attn_ckpt_version='mix',
    weight_dtype=torch.float16,
    use_tf32=True,
    device='cuda',
    skip_safety_check=True
)

print('⚡ Initializing AutoMasker (DensePose + SCHP) for Pose & Clothing Segmentation...')
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)
automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, 'DensePose'),
    schp_ckpt=os.path.join(repo_path, 'SCHP'),
    device='cuda'
)

print('✅ CatVTON & AutoMasker successfully loaded in GPU!')
print(f'🔥 VRAM in use: {round(torch.cuda.memory_allocated() / (1024**3), 2)} GB / 15.0 GB')


### 🌐 Step 4: Launch FastAPI Server & Cloudflare Tunnel
Starts the HTTP API on port 8000 and tunnels it to an HTTPS public URL. Copy the resulting URL into your `backend/.env` file!

In [ ]:
# @title 4. Start Server & Generate Public URL (Keep this cell running!)
import os
import sys
import subprocess

# Auto-detect or clone CatVTON directory
work_dir = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
catvton_dir = os.path.join(work_dir, 'CatVTON')
if not os.path.exists(catvton_dir):
    if os.path.exists(os.path.join(os.getcwd(), 'CatVTON')):
        catvton_dir = os.path.join(os.getcwd(), 'CatVTON')
    elif os.path.exists('model') and os.path.exists('utils.py'):
        catvton_dir = os.getcwd()
    else:
        print('📥 CatVTON directory missing, cloning...')
        subprocess.run(['git', 'clone', 'https://github.com/Zheng-Chong/CatVTON.git', catvton_dir])

if catvton_dir not in sys.path:
    sys.path.insert(0, catvton_dir)
if os.path.exists(catvton_dir):
    os.chdir(catvton_dir)

import io
import time
import re
import socket
import subprocess
import threading
import uvicorn
from PIL import Image
from utils import resize_and_crop, resize_and_padding
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse

# Compatibility patches for transformers & accelerate
try:
    import transformers
    if not hasattr(transformers, 'EncoderDecoderCache'):
        class EncoderDecoderCache:
            pass
        transformers.EncoderDecoderCache = EncoderDecoderCache
except Exception:
    pass

try:
    import accelerate.utils
    import accelerate.utils.memory
    def _clear_device_cache():
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if not hasattr(accelerate.utils.memory, 'clear_device_cache'):
        accelerate.utils.memory.clear_device_cache = _clear_device_cache
    if not hasattr(accelerate.utils, 'clear_device_cache'):
        accelerate.utils.clear_device_cache = _clear_device_cache
except Exception:
    pass

app = FastAPI(title='StyleVision CatVTON GPU Server')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.get('/')
@app.get('/health')
def health():
    return {
        'status': 'online',
        'service': 'StyleVision CatVTON GPU Server',
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'vram_allocated_gb': round(torch.cuda.memory_allocated() / (1024**3), 2)
    }

@app.post('/tryon')
async def tryon(
    person_image: UploadFile = File(..., description='Full person photo (pose is preserved)'),
    cloth_image: UploadFile = File(..., description='Target clothing image (saree, dress, gown, top)'),
    cloth_type: str = Form('overall', description='"overall" | "upper" | "lower"'),
    num_steps: int = Form(40, description='Inference steps (30-50)'),
    guidance_scale: float = Form(2.5, description='CFG scale'),
    seed: int = Form(42, description='Seed for reproducibility')
):
    try:
        if 'automasker' not in globals() or 'pipeline' not in globals():
            raise HTTPException(status_code=503, detail='Models not loaded. Please run Cell 3 first before Cell 4.')
        p_bytes = await person_image.read()
        c_bytes = await cloth_image.read()
        person_pil = Image.open(io.BytesIO(p_bytes)).convert('RGB')
        cloth_pil = Image.open(io.BytesIO(c_bytes)).convert('RGB')

        target_w, target_h = 768, 1024
        person_resized = resize_and_crop(person_pil, (target_w, target_h))
        cloth_resized = resize_and_padding(cloth_pil, (target_w, target_h))

        # Standard AutoMasker (preserves original body shape, face, pose, and background)
        mask = automasker(person_resized, cloth_type)['mask']
        mask = mask_processor.blur(mask, blur_factor=9)

        generator = torch.Generator(device='cuda').manual_seed(seed) if seed != -1 else None
        with torch.inference_mode():
            result = pipeline(
                image=person_resized,
                condition_image=cloth_resized,
                mask=mask,
                num_inference_steps=num_steps,
                guidance_scale=guidance_scale,
                generator=generator
            )[0]

        buf = io.BytesIO()
        result.save(buf, format='PNG')
        buf.seek(0)
        return StreamingResponse(buf, media_type='image/png', headers={'X-TryOn-Status': 'success'})
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

def get_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

port = get_free_port()
print(f'🚀 Starting FastAPI server on port {port}...')

# Kill any previous cloudflared processes
subprocess.run(['pkill', '-9', '-f', 'cloudflared'], stderr=subprocess.DEVNULL)
time.sleep(1)

def run_server():
    import asyncio
    asyncio.set_event_loop_policy(asyncio.DefaultEventLoopPolicy())
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host='127.0.0.1', port=port, log_level='warning', loop='asyncio')
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

work_dir = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
log_path = os.path.join(work_dir, 'cloudflared.log')
if os.path.exists(log_path):
    os.remove(log_path)

print('🌐 Starting Cloudflare Tunnel to expose the API publicly...')
log_file = open(log_path, 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
    stdout=log_file,
    stderr=log_file
)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 35:
    time.sleep(1)
    if os.path.exists(log_path):
        with open(log_path, 'r') as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print('\n' + '='*75)
    print('🎉 STYLEVISION CatVTON GPU SERVER IS RUNNING!')
    print(f'👉 PUBLIC URL: {tunnel_url}')
    print('='*75)
    print('🟢 Server is active. KEEP THIS CELL RUNNING to stay connected!')
    try:
        while True:
            time.sleep(10)
    except KeyboardInterrupt:
        print('Stopping tunnel...')
        tunnel_proc.terminate()
else:
    print('❌ Cloudflare Tunnel timed out. Check /kaggle/working/cloudflared.log')


### 🎨 Step 5 (Optional): Test in Browser with Gradio UI
Want to test right in your browser? Run this cell to get an interactive Gradio web app with side-by-side try-on preview!

In [ ]:
# @title 5. Interactive In-Browser Demo (Optional)
import gradio as gr

def interactive_tryon(person_img, cloth_img, cloth_type):
    if person_img is None or cloth_img is None:
        return None
    p = person_img.convert('RGB')
    c = cloth_img.convert('RGB')
    p_res = resize_and_crop(p, (768, 1024))
    c_res = resize_and_padding(c, (768, 1024))
    mask = automasker(p_res, cloth_type)['mask']
    mask = mask_processor.blur(mask, blur_factor=9)
    with torch.inference_mode():
        result = pipeline(image=p_res, condition_image=c_res, mask=mask, num_inference_steps=40, guidance_scale=2.5)[0]
    return result

demo = gr.Interface(
    fn=interactive_tryon,
    inputs=[
        gr.Image(label='Target Person Photo (Pose Preserved)', type='pil'),
        gr.Image(label='Garment (Saree / Dress / Gown / Kurti)', type='pil'),
        gr.Radio(['overall', 'upper', 'lower'], value='overall', label='Cloth Type')
    ],
    outputs=gr.Image(label='Synthesized Output (Pose Kept)'),
    title='StyleVision CatVTON Interactive Studio'
)
demo.queue().launch(share=True)